In [4]:
#micpas自动下载tlogp数据
import os
import requests
import datetime

def download_diamond(start_time, source_path, out_path):
    """
    下载micaps格式数据，保存文件为二进制格式
    :param start_time:  起报时间
    :param source_path: mdfs数据路径
    :param out_path: 文件输出路径
    :return: None
    """
    start_time_str = start_time.strftime("%Y%m%d%H") + "0000"
    if not os.path.exists(out_path): 
        os.makedirs(out_path)
    
    filename = f"{start_time_str[:]}"+".000"
    out_filename = os.path.join(out_path, filename)
    # 10.xx.xx.146为mdfs数据源地址，需修改
    url = f"http://10.172.10.38:8080/DataService?requestType=getData&directory={source_path}&fileName={filename}"
    print(out_filename)
    re = requests.get(url)
    content = re.content
    
    with open(out_filename, "wb") as fw:
        fw.write(content[14:])

def get_past_month_dates():
    """
    获取过去一个月的日期列表，包含08:00和20:00两个时间点
    :return: 日期时间列表
    """
    today = datetime.date.today()
    dates = []
    for i in range(1):  # 过去30天
        date = today - datetime.timedelta(days=i)
        dates.append(datetime.datetime(date.year, date.month, date.day, 8))   # 08:00
        dates.append(datetime.datetime(date.year, date.month, date.day, 20))  # 20:00
    return dates

if __name__ == '__main__':
    source_path = "UPPER_AIR/TLOGP"
    out_path = r"E:/data/"  # 保存数据的位置
    
    # 获取过去一个月的所有日期时间点
    date_times = get_past_month_dates()
    
    # 下载每个日期时间点的数据
    for start_time in date_times:
        download_diamond(start_time, source_path, out_path)


E:/data/20250303080000.000
E:/data/20250303200000.000


In [9]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import metpy.calc as mpcalc
from metpy.plots import Hodograph, SkewT
from metpy.units import units
import metpy



# 定义获取文件路径列表的函数
def get_files_path_list(path):
    files = []
    times = []
    filesList = os.listdir(path)
    for filename in filesList:
        if os.path.splitext(filename)[1] == '.000': 
            fileAbsPath = os.path.join(path, filename)
            files.append(fileAbsPath)
            times.append(filename[0:10])
    return files, times

# 设置路径
path = r'E:\data'  # tlogp资料存放路径

# 定义读取和处理文件的函数
def duqu(file, zh, t):
    pathsave = f'E:/TLOGP/1.txt'  # 提取数据保存路径
    
    # 创建并写入文件头
    with open(pathsave, "a+", encoding='utf-8') as f:
        f.write("'pressure', 'height', 'temperature', 'dewpoint', 'direction', 'speed','date'")
        f.write("\n")

    for ii in range(len(file)):
        ss = []
        ss1 = []
        df = pd.read_csv(file[ii], header=None, sep="   ", skiprows=2, names=["h1", "h2", "h3", "h4", "h5", "h6"])
        r = df.loc[df["h1"].str.contains('57131')]  # 查找探空站号
        
        if len(r) > 0:
            rr = r.index.tolist()
            r1 = int(rr[0])
            
            for i in range(1, 500):
                if len(df.iloc[r1 + i].h1) > 10:
                    break
                else:
                    ss = df.iloc[r1 + i].tolist()
                    ss.append(str(t[ii]))
                    ss1.append(ss)
            
            df1 = pd.DataFrame(ss1)
            
            # 保存数据到文件
            if ii == 0:
                df1.to_csv(pathsave, index=False, mode="a+", header=["press", "hgt", "t", "td", "w", "s", "date"])
            else:
                df1.to_csv(pathsave, index=False, mode="a+", header=False)

# 获取文件路径列表和时间信息
files_path, times = get_files_path_list(path)

# 调用处理函数
duqu(files_path, "", times)


C:\Users\xiaomi\AppData\Local\Temp\ipykernel_2352\170275861.py:38: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(file[ii], header=None, sep="   ", skiprows=2, names=["h1", "h2", "h3", "h4", "h5", "h6"])


In [10]:


# 定义列名称
col_names = ['pressure', 'height', 'temperature', 'dewpoint', 'direction', 'speed', "time"]

# 读取文件，注意修正 usecols 参数中的列索引
df = pd.read_csv('E:/TLOGP/1.txt',
                skiprows=2,  # 跳过前四行
                names=col_names)  # 直接使用列名称

# 显示数据框的前五行

df = df.dropna(subset=('temperature', 'dewpoint', 'direction', 'speed'), how='all').reset_index(drop=True)
df


,pressure,height,temperature,dewpoint,direction,speed,time
0,959.3,41.1,28.2,23.1,71.0,3.3,2021081920
1,959.3,41.1,28.2,23.1,71.0,3.3,2021081920
2,959.3,41.1,28.2,23.1,71.0,3.2,2021081920
3,958.0,41.6,27.5,22.9,70.0,3.2,2021081920
4,941.7,57.6,26.3,22.1,48.0,2.9,2021081920
...,...,...,...,...,...,...,...
124,48.5,2109.4,-62.8,9999.0,84.0,8.9,2021081920
125,47.2,2120.6,-61.2,9999.0,84.0,8.9,2021081920
126,46.2,2133.2,-61.7,9999.0,84.0,8.9,2021081920
127,45.7,2152.6,-60.3,9999.0,84.0,8.9,2021081920


In [11]:

import numpy as np
import pandas as pd

# 定义列名称
col_names = ['pressure', 'height', 'temperature', 'dewpoint', 'direction', 'speed', "time"]

# 读取文件，注意修正 usecols 参数中的列索引
df = pd.read_csv('E:/TLOGP/1.txt',
                skiprows=2,  # 跳过前四行
                names=col_names, 
                
                dtype={'pressure':np.float32, 'height':np.float32, 'temperature':np.float32, 'dewpoint':np.float32, 'direction':np.float32, 'speed':np.float32},)  # 直接使用列名称

# 显示数据框的前五行


In [12]:
df['dewpoint'].replace(9999, 0, inplace=True)
df['direction'].replace(9999, 0, inplace=True)
df['temperature'].replace(9999, 0, inplace=True)

p = df['pressure'].values * units.hPa                # 单位：hPa
T = df['temperature'].values * units.degC            # 单位：℃
Td = df['dewpoint'].values * units.degC              # 单位：℃
wind_speed = df['speed'].values * units.knots        # 单位：knot
wind_dir = df['direction'].values * units.degrees    # 单位：°
u, v = mpcalc.wind_components(wind_speed, wind_dir)  # 计算水平风速u和v

C:\Users\xiaomi\AppData\Local\Temp\ipykernel_2352\1056137914.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['dewpoint'].replace(9999, 0, inplace=True)
C:\Users\xiaomi\AppData\Local\Temp\ipykernel_2352\1056137914.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exampl

In [13]:
def calculate_indices(pressure, temperature, dewpoint, height):
    import numpy as np
    import metpy.calc as mpcalc
    from metpy.units import units
    
    # LCL 计算
    lcl_pressure, lcl_temperature = mpcalc.lcl(pressure[0], temperature[0], dewpoint[0])

    # CAPE 和 CIN 计算
    parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')
    cape, cin = mpcalc.cape_cin(pressure, temperature, dewpoint, parcel_prof)

    # DCAPE 计算
    dcape = mpcalc.cape_cin(pressure[::-1], temperature[::-1], dewpoint[::-1], parcel_prof[::-1])[0]

    # SI 指数计算
    si_index = (temperature[pressure == 500 * units.hPa][0] - 
                parcel_prof[pressure == 500 * units.hPa][0]).magnitude

    # K 指数计算
    k_index = ((temperature[pressure == 850 * units.hPa][0] - 
                dewpoint[pressure == 850 * units.hPa][0]) + 
                dewpoint[pressure == 700 * units.hPa][0] - 
                (temperature[pressure == 700 * units.hPa][0] - 
                 temperature[pressure == 500 * units.hPa][0])).magnitude

    # 计算 0℃ 层高度
    zero_level_idx = np.where(temperature <= 0 * units.degC)[0]
    zero_level_height = height[zero_level_idx[0]].m if len(zero_level_idx) > 0 else np.nan

    # 返回计算结果
    return {
        "LCL Pressure": lcl_pressure,
        "LCL Temperature": lcl_temperature,
        "CAPE": cape,
        "CIN": cin,
        "DCAPE": dcape,
        "SI Index": si_index,
        "K Index": k_index,
        "0℃层高度": zero_level_height  
    }


In [14]:
# 计算探空参数
def calculate_indices(pressure, temperature, dewpoint):
    # 1. LCL（抬升凝结高度）
    lcl_pressure, lcl_temperature = mpcalc.lcl(pressure[0], temperature[0], dewpoint[0])

    # 2. CAPE 和 CIN
    parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')
    cape, cin = mpcalc.cape_cin(pressure, temperature, dewpoint, parcel_prof)

    # 3. DCAPE（下降对流有效位能） - 这里需注意数据范围
    dcape = mpcalc.cape_cin(pressure[::-1], temperature[::-1], dewpoint[::-1], parcel_prof[::-1])[0]

    # 4. SI指数（Showalter Index）
    si_index = (temperature[pressure == 500 * units.hPa][0] -
                parcel_prof[pressure == 500 * units.hPa][0]).magnitude

    # 5. K指数
    k_index = ((temperature[pressure == 850 * units.hPa][0] -
                dewpoint[pressure == 850 * units.hPa][0]) +
               dewpoint[pressure == 700 * units.hPa][0] -
               (temperature[pressure == 700 * units.hPa][0] -
                temperature[pressure == 500 * units.hPa][0])).magnitude

    return {
        "LCL Pressure": lcl_pressure,
        "LCL Temperature": lcl_temperature,
        "CAPE": cape,
        "CIN": cin,
        "DCAPE": dcape,
        "SI Index": si_index,
        "K Index": k_index
    }

In [15]:
calculate_indices(p, T, Td)

C:\Users\xiaomi\AppData\Local\Temp\ipykernel_2352\916801282.py:7: UserWarning: Duplicate pressure(s) [45.70000076293945 83.0999984741211 103.30000305175781 119.5999984741211] hPa provided. Output profile includes duplicate temperatures as a result.
  parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')


{'LCL Pressure': <Quantity(890.7133178710938, 'hectopascal')>,
 'LCL Temperature': <Quantity(21.880218505859375, 'degree_Celsius')>,
 'CAPE': <Quantity(1279.06251, 'joule / kilogram')>,
 'CIN': <Quantity(-38.4635888, 'joule / kilogram')>,
 'DCAPE': <Quantity(0.0, 'joule / kilogram')>,
 'SI Index': np.float64(-3.1797587275958676),
 'K Index': np.float32(-1.0)}

In [24]:
import numpy as np
import metpy.calc as mpcalc
from metpy.units import units

def calculate_indices(pressure, temperature, dewpoint, height):
    """计算常用的对流指数"""
    lcl_pressure, lcl_temperature = mpcalc.lcl(pressure[0], temperature[0], dewpoint[0])
    parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')
    cape, cin = mpcalc.cape_cin(pressure, temperature, dewpoint, parcel_prof)
    dcape = mpcalc.cape_cin(pressure[::-1], temperature[::-1], dewpoint[::-1], parcel_prof[::-1])[0]
    
    # Showalter Index (SI)
    si_index = ((temperature[pressure == 500 * units.hPa][0] - 
                 parcel_prof[pressure == 500 * units.hPa][0]).magnitude)

    # K Index
    k_index = ((temperature[pressure == 850 * units.hPa][0] - dewpoint[pressure == 850 * units.hPa][0]) + 
               dewpoint[pressure == 700 * units.hPa][0] - 
               (temperature[pressure == 700 * units.hPa][0] - temperature[pressure == 500 * units.hPa][0])).magnitude

    # 计算0℃层高度
    zero_level_idx = np.where(temperature <= 0 * units.degC)[0]
    zero_level_height = height[zero_level_idx[0]].m if len(zero_level_idx) > 0 else np.nan

    return {
        "LCL Pressure": lcl_pressure,
        "LCL Temperature": lcl_temperature,
        "CAPE": cape,
        "CIN": cin,
        "DCAPE": dcape,
        "SI Index": si_index,
        "K Index": k_index,
        "0℃层高度": zero_level_height
    }

def check_convective_alert(indices):
    """改进预警逻辑，单个阈值超过即报警"""
    alerts = []

    # **单项指标触发预警**
    if indices["CAPE"] > 800 * units("J/kg"):
        alerts.append(f"⚠ CAPE过高 ({indices['CAPE'].magnitude:.1f} J/kg)")
    if indices["K Index"] > 36:
        alerts.append(f"⚠ K指数过高 ({indices['K Index']:.1f}℃)")
    if indices["SI Index"] < 0:
        alerts.append(f"⚠ SI指数过低 ({indices['SI Index']:.1f}℃)")
    if indices["CIN"] < 50 * units("J/kg"):
        alerts.append(f"⚠ CIN值较低 ({indices['CIN'].magnitude:.1f} J/kg)，有利于对流触发")
    if 5100 <= indices["0℃层高度"] <= 5500:
        alerts.append(f"⚠ 0℃层高度适中 ({indices['0℃层高度']:.1f}m)，有利于强对流发展")

    # **强对流和稳定性暴雨判定**
    convective_rain = any([
        indices["CAPE"] > 800 * units("J/kg"),
        indices["K Index"] > 36,
        indices["SI Index"] < 0,
        indices["CIN"] < 50 * units("J/kg"),
        5100 <= indices["0℃层高度"] <= 5500
    ])

    stable_rain = any([
        indices["CAPE"] < 100 * units("J/kg"),
        indices["K Index"] < 36,
        indices["SI Index"] > 0,
        indices["CIN"] > 50 * units("J/kg"),
        4800 <= indices["0℃层高度"] <= 5100
    ])

    if convective_rain:
        alerts.append("🚨 可能出现**对流性暴雨**")
    if stable_rain:
        alerts.append("🌧 可能出现**稳定性降水**")

    return alerts

if __name__ == "__main__":
    # 数据预处理（确保单位正确）
    p = df['pressure'].values * units.hPa
    T = df['temperature'].values * units.degC
    Td = df['dewpoint'].values * units.degC
    height = df['height'].values * units.meter  

    # 计算指数
    indices = calculate_indices(p, T, Td, height)

    # 触发预警
    alerts = check_convective_alert(indices)

    # 输出结果
    if alerts:
        print("\n".join(alerts))
    else:
        print("✅ 当前无强对流风险")


⚠ CAPE过高 (1279.1 J/kg)
⚠ SI指数过低 (-3.2℃)
⚠ CIN值较低 (-38.5 J/kg)，有利于对流触发
🚨 可能出现**对流性暴雨**
🌧 可能出现**稳定性降水**


C:\Users\xiaomi\AppData\Local\Temp\ipykernel_2352\3156589411.py:8: UserWarning: Duplicate pressure(s) [45.70000076293945 83.0999984741211 103.30000305175781 119.5999984741211] hPa provided. Output profile includes duplicate temperatures as a result.
  parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')


In [21]:
def calculate_indices(pressure, temperature, dewpoint, height):
    """计算常用的对流指数"""
    lcl_pressure, lcl_temperature = mpcalc.lcl(pressure[0], temperature[0], dewpoint[0])
    parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')
    cape, cin = mpcalc.cape_cin(pressure, temperature, dewpoint, parcel_prof)
    dcape = mpcalc.cape_cin(pressure[::-1], temperature[::-1], dewpoint[::-1], parcel_prof[::-1])[0]
    
    # Showalter Index (SI)
    si_index = ((temperature[pressure == 500 * units.hPa][0] - 
                 parcel_prof[pressure == 500 * units.hPa][0]).magnitude)

    # K Index
    k_index = ((temperature[pressure == 850 * units.hPa][0] - dewpoint[pressure == 850 * units.hPa][0]) + 
               dewpoint[pressure == 700 * units.hPa][0] - 
               (temperature[pressure == 700 * units.hPa][0] - temperature[pressure == 500 * units.hPa][0])).magnitude

    # 计算0℃层高度
    zero_level_idx = np.where(temperature <= 0 * units.degC)[0]
    zero_level_height = height[zero_level_idx[0]].m if len(zero_level_idx) > 0 else np.nan

    return {
        "LCL Pressure": lcl_pressure,
        "LCL Temperature": lcl_temperature,
        "CAPE": cape,
        "CIN": cin,
        "DCAPE": dcape,
        "SI Index": si_index,
        "K Index": k_index,
        "0℃层高度": zero_level_height
    }


def check_convective_alert(indices):
    """最终预警判断，只有满足所有条件才会触发"""
    criteria_met = {
        "对流性暴雨": (
            indices["CAPE"] > 800 * units("J/kg"),
            indices["K Index"] > 36,
            indices["SI Index"] < 0,
            indices["CIN"] < 50 * units("J/kg"),
            5100 <= indices["0℃层高度"] <= 5500
        ),
        "稳定性暴雨": (
            indices["CAPE"] < 100 * units("J/kg"),
            indices["K Index"] < 36,
            indices["SI Index"] > 0,
            indices["CIN"] > 50 * units("J/kg"),
            4800 <= indices["0℃层高度"] <= 5100
        )
    }

    alert = [storm_type for storm_type, conditions in criteria_met.items() if all(conditions)]
    
    return " | ".join(alert) if alert else "无强对流条件"


if __name__ == "__main__":
    # 数据预处理（确保单位正确）
    p = df['pressure'].values * units.hPa
    T = df['temperature'].values * units.degC
    Td = df['dewpoint'].values * units.degC
    height = df['height'].values * units.meter  

    # 计算指数
    indices = calculate_indices(p, T, Td, height)

    # **预警条件触发检查**
    preliminary_alerts = []
    if indices["CAPE"] > 100 * units("J/kg"):
        preliminary_alerts.append(f"CAPE值较高（{indices['CAPE'].m:.1f} J/kg）")
    if indices["K Index"] > 32:
        preliminary_alerts.append(f"K指数较高（{indices['K Index']:.1f}℃）")
    if indices["SI Index"] < 0:
        preliminary_alerts.append(f"SI指数较低（{indices['SI Index']:.1f}℃）")
    
    # **最终预警判断**
    alert = check_convective_alert(indices)
    
    # **优化报警逻辑**
    if preliminary_alerts:
        print("⚠ 预警信号触发:", ", ".join(preliminary_alerts))
    
    if alert != "无强对流条件":
        print("🚨 强对流天气预警:", alert)
    else:
        print("✅ 当前无强对流风险")


⚠ 预警信号触发: CAPE值较高（1279.1 J/kg）, SI指数较低（-3.2℃）
✅ 当前无强对流风险


C:\Users\xiaomi\AppData\Local\Temp\ipykernel_2352\1800648396.py:4: UserWarning: Duplicate pressure(s) [45.70000076293945 83.0999984741211 103.30000305175781 119.5999984741211] hPa provided. Output profile includes duplicate temperatures as a result.
  parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')


In [16]:
# 修改后的对流预警判断代码
def calculate_indices(pressure, temperature, dewpoint, height):
    # 1. LCL（抬升凝结高度）
    lcl_pressure, lcl_temperature = mpcalc.lcl(pressure[0], temperature[0], dewpoint[0])

    # 2. CAPE 和 CIN
    parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')
    cape, cin = mpcalc.cape_cin(pressure, temperature, dewpoint, parcel_prof)

    # 3. DCAPE（下降对流有效位能）
    dcape = mpcalc.cape_cin(pressure[::-1], temperature[::-1], dewpoint[::-1], parcel_prof[::-1])[0]

    # 4. SI指数（Showalter Index）
    si_index = (temperature[pressure == 500 * units.hPa][0] - 
                parcel_prof[pressure == 500 * units.hPa][0]).magnitude

    # 5. K指数
    k_index = ((temperature[pressure == 850 * units.hPa][0] - 
               dewpoint[pressure == 850 * units.hPa][0]) + 
               dewpoint[pressure == 700 * units.hPa][0] - 
               (temperature[pressure == 700 * units.hPa][0] - 
                temperature[pressure == 500 * units.hPa][0])).magnitude

    # 6. 计算0℃层高度
    zero_level_idx = np.where(temperature <= 0 * units.degC)[0]
    zero_level_height = height[zero_level_idx[0]].m if len(zero_level_idx) > 0 else np.nan

    return {
        "LCL Pressure": lcl_pressure,
        "LCL Temperature": lcl_temperature,
        "CAPE": cape,
        "CIN": cin,
        "DCAPE": dcape,
        "SI Index": si_index,
        "K Index": k_index,
        "0℃层高度": zero_level_height
    }

def check_convective_alert(indices):
    criteria_met = {
        "对流性暴雨": (
            indices["CAPE"] > 800 * units("J/kg"),
            indices["K Index"] > 36,
            indices["SI Index"] < 0,
            indices["CIN"] < 50 * units("J/kg"),
            5100 <= indices["0℃层高度"] <= 5500
        ),
        "稳定性暴雨": (
            indices["CAPE"] < 100 * units("J/kg"),
            indices["K Index"] < 36,
            indices["SI Index"] > 0,
            indices["CIN"] > 50 * units("J/kg"),
            4800 <= indices["0℃层高度"] <= 5100
        )
    }

    alert = []
    for storm_type, conditions in criteria_met.items():
        if all(conditions):
            alert.append(storm_type)
    
    return " | ".join(alert) if alert else "无强对流条件"

# 主程序流程
if __name__ == "__main__":
    # 数据预处理（确保单位正确）
    p = df['pressure'].values * units.hPa
    T = df['temperature'].values * units.degC
    Td = df['dewpoint'].values * units.degC
    height = df['height'].values * units.meter  # 添加高度参数

    # 计算指数
    indices = calculate_indices(p, T, Td, height)

    # 初步对流天气判断
    preliminary_alerts = []
    if indices["CAPE"] > 100 * units("J/kg"):
        preliminary_alerts.append(f"CAPE值过高（{indices['CAPE'].m:.1f} J/kg）")
    if indices["K Index"] > 32:
        preliminary_alerts.append(f"K指数过高（{indices['K Index']:.1f}℃）")
    if indices["SI Index"] < 0:
        preliminary_alerts.append(f"SI指数过低（{indices['SI Index']:.1f}℃）")
    
    # 最终预警判断
    alert = check_convective_alert(indices)
    
    # 输出结果
    if preliminary_alerts:
        print("对流预警条件触发:", ", ".join(preliminary_alerts))
    print("对流预警结果:", alert)

对流预警条件触发: CAPE值过高（1279.1 J/kg）, SI指数过低（-3.2℃）
对流预警结果: 无强对流条件


C:\Users\xiaomi\AppData\Local\Temp\ipykernel_2352\1886011678.py:7: UserWarning: Duplicate pressure(s) [45.70000076293945 83.0999984741211 103.30000305175781 119.5999984741211] hPa provided. Output profile includes duplicate temperatures as a result.
  parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')


In [18]:
# 修改后的对流预警判断代码
def calculate_indices(pressure, temperature, dewpoint, height):
    # 1. LCL（抬升凝结高度）
    lcl_pressure, lcl_temperature = mpcalc.lcl(pressure[0], temperature[0], dewpoint[0])

    # 2. CAPE 和 CIN
    parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')
    cape, cin = mpcalc.cape_cin(pressure, temperature, dewpoint, parcel_prof)

    # 3. DCAPE（下降对流有效位能）
    dcape = mpcalc.cape_cin(pressure[::-1], temperature[::-1], dewpoint[::-1], parcel_prof[::-1])[0]

    # 4. SI指数（Showalter Index）
    si_index = (temperature[pressure == 500 * units.hPa][0] - 
                parcel_prof[pressure == 500 * units.hPa][0]).magnitude

    # 5. K指数
    k_index = ((temperature[pressure == 850 * units.hPa][0] - 
               dewpoint[pressure == 850 * units.hPa][0]) + 
               dewpoint[pressure == 700 * units.hPa][0] - 
               (temperature[pressure == 700 * units.hPa][0] - 
                temperature[pressure == 500 * units.hPa][0])).magnitude

    # 6. 计算0℃层高度
    zero_level_idx = np.where(temperature <= 0 * units.degC)[0]
    zero_level_height = height[zero_level_idx[0]].m if len(zero_level_idx) > 0 else np.nan

    return {
        "LCL Pressure": lcl_pressure,
        "LCL Temperature": lcl_temperature,
        "CAPE": cape,
        "CIN": cin,
        "DCAPE": dcape,
        "SI Index": si_index,
        "K Index": k_index,
        "0℃层高度": zero_level_height
    }

def check_convective_alert(indices):
    criteria_met = {
        "对流性暴雨": (
            indices["CAPE"] > 800 * units("J/kg"),
            indices["K Index"] > 36,
            indices["SI Index"] < 0,
            indices["CIN"] < 50 * units("J/kg"),
            5100 <= indices["0℃层高度"] <= 5500
        ),
        "稳定性暴雨": (
            indices["CAPE"] < 100 * units("J/kg"),
            indices["K Index"] < 36,
            indices["SI Index"] > 0,
            indices["CIN"] > 50 * units("J/kg"),
            4800 <= indices["0℃层高度"] <= 5100
        )
    }

    alert = []
    for storm_type, conditions in criteria_met.items():
        if all(conditions):
            alert.append(storm_type)
    
    return " | ".join(alert) if alert else "无强对流条件"

# 主程序流程
if __name__ == "__main__":
    # 数据预处理（确保单位正确）
    p = df['pressure'].values * units.hPa
    T = df['temperature'].values * units.degC
    Td = df['dewpoint'].values * units.degC
    height = df['height'].values * units.meter  # 添加高度参数

    # 计算指数
    indices = calculate_indices(p, T, Td, height)

    # 初步对流天气判断
    preliminary_alerts = []
    if indices["CAPE"] > 100 * units("J/kg"):
        preliminary_alerts.append(f"CAPE值过高（{indices['CAPE'].m:.1f} J/kg）")
    if indices["K Index"] > 32:
        preliminary_alerts.append(f"K指数过高（{indices['K Index']:.1f}℃）")
    if indices["SI Index"] < 0:
        preliminary_alerts.append(f"SI指数过低（{indices['SI Index']:.1f}℃）")
    
    # 最终预警判断
    alert = check_convective_alert(indices)
    
    # 输出结果
    if preliminary_alerts:
        print("对流预警条件触发:", ", ".join(preliminary_alerts))
    print("对流预警结果:", alert)

对流预警条件触发: CAPE值过高（1279.1 J/kg）, SI指数过低（-3.2℃）
对流预警结果: 无强对流条件


C:\Users\xiaomi\AppData\Local\Temp\ipykernel_2352\1886011678.py:7: UserWarning: Duplicate pressure(s) [45.70000076293945 83.0999984741211 103.30000305175781 119.5999984741211] hPa provided. Output profile includes duplicate temperatures as a result.
  parcel_prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')
